<a href="https://colab.research.google.com/github/Isshoo/model-pendeteksi-jajanan/blob/main/Model_Jajanan_Manado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Mount Google Drive (if your dataset is there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/dataset_daun_jagung/"
train_dir = os.path.join(DATASET_PATH, "train")
test_dir = os.path.join(DATASET_PATH, "val")

In [ ]:
import matplotlib.image as mpimg

plt.imshow(mpimg.imread(os.path.join(train_dir, "Bercak_Daun/0a01cc10-3892-4311-9c48-0ac6ab3c7c43___RS_GLSp 9352_90deg")))
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# All images will be rescaled by 1./255
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,           # Kurangi dari 40
    width_shift_range=0.15,      # Kurangi dari 0.2
    height_shift_range=0.15,     # Kurangi dari 0.2
    shear_range=0.15,           # Kurangi dari 0.2
    zoom_range=0.15,            # Kurangi dari 0.2
    horizontal_flip=True,
    vertical_flip=True,         # Tambahkan vertical flip
    brightness_range=[0.8, 1.2], # Tambahkan brightness
    fill_mode='nearest',
    # Tambahkan channel shift
    channel_shift_range=0.1
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

In [ ]:
# Perbaikan untuk menampilkan augmented images
def plotImages(images_arr):
    fig, axes = plt.subplots(1, len(images_arr), figsize=(20, 4))
    if len(images_arr) == 1:
        axes = [axes]

    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Cara yang benar untuk mendapatkan augmented images
sample_batch = next(train_generator)
augmented_images = sample_batch[0][:5]  # Ambil 5 gambar pertama
plotImages(augmented_images)

In [ ]:
validation_datagen = ImageDataGenerator(
    rescale=1./255
)
validation_generator = validation_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
print("Training classes found:", train_generator.class_indices)
print("Number of training samples:", train_generator.samples)
print("Number of validation samples:", validation_generator.samples)
print("Number of classes:", train_generator.num_classes)

In [ ]:
# Model dengan lebih banyak layer dan batch normalization
model = tf.keras.models.Sequential([
    # Block 1
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Dropout(0.25),

    # Block 2
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Dropout(0.25),

    # Block 3
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Dropout(0.25),

    # Block 4
    tf.keras.layers.Conv2D(256, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # Classifier
    tf.keras.layers.GlobalAveragePooling2D(),  # Lebih baik dari Flatten
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation='softmax')
])

model.summary()

In [ ]:
# Gunakan pre-trained model ()
base_model = tf.keras.applications.DenseNet121(
    input_shape=(224, 224, 3),  # Ukuran input lebih besar
    include_top=False,
    weights='imagenet'
)

# Freeze base model layers
base_model.trainable = False

In [ ]:
# Tambahkan custom classifier
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation='softmax')
])

In [ ]:
# Gunakan learning rate scheduling
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# Learning rate yang adaptif
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Early stopping untuk mencegah overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Simpan model terbaik
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Compile dengan optimizer yang lebih baik
model.compile(
    optimizer=Adam(learning_rate=0.001),  # Learning rate lebih tinggi di awal
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==================== STRATEGI 5: TRAINING DENGAN CALLBACKS ====================

# Training dengan lebih banyak epochs dan callbacks
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=50,  # Lebih banyak epochs
    steps_per_epoch=len(train_generator),  # Gunakan semua data
    validation_steps=len(validation_generator),
    callbacks=[reduce_lr, early_stop, checkpoint],
    verbose=1
)


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = np.arange(len(history.history['accuracy']))

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

In [ ]:
print("="*50)
print("MODEL EVALUATION")
print("="*50)

# 1. Evaluasi pada test set
print("Evaluating model on test set...")
test_loss, test_accuracy = model.evaluate(validation_generator, verbose=1)
print(f"\nTest Results:")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Loss: {test_loss:.4f}")

print("="*50)
print("EVALUATION COMPLETE")
print("="*50)

In [ ]:
# Setelah training pertama, unfreeze beberapa layer terakhir
base_model.trainable = True

# Fine-tune hanya layer terakhir
fine_tune_at = 100  # Freeze layer sebelum ini

# Freeze layer awal
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Compile ulang dengan learning rate yang sangat kecil
model.compile(
    optimizer=Adam(learning_rate=0.0001/10),  # Learning rate 10x lebih kecil
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tuning training
fine_tune_epochs = 20
total_epochs = len(history.history['accuracy']) + fine_tune_epochs

history_fine = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=total_epochs,
    initial_epoch=len(history.history['accuracy']),
    steps_per_epoch=len(train_generator),
    validation_steps=len(validation_generator),
    callbacks=[reduce_lr, early_stop, checkpoint],
    verbose=1
)

In [ ]:
import numpy as np
from google.colab import files
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
from PIL import Image
import io

# Upload file
uploaded = files.upload()

# Dapatkan class names dari generator
class_names = list(train_generator.class_indices.keys())
print("Class names:", class_names)

for fn in uploaded.keys():
    path = '/content/' + fn

    # Preview gambar yang diupload
    img_display = Image.open(path)
    plt.figure(figsize=(8, 6))
    plt.imshow(img_display)
    plt.title(f'Uploaded Image: {fn}')
    plt.axis('off')
    plt.show()

    # Load dan preprocess gambar untuk prediksi
    # PERBAIKAN: Ubah target_size menjadi (224, 224) sesuai dengan yang diharapkan model
    img = image.load_img(path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = x / 255.0  # Normalisasi
    x = np.expand_dims(x, axis=0)

    # Prediksi
    predictions = model.predict(x)
    predicted_class_index = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_index]

    # Tampilkan hasil
    print(f"File: {fn}")
    print(f"Predicted class: {class_names[predicted_class_index]}")
    print(f"Confidence: {confidence:.4f}")
    print(f"All probabilities: {predictions[0]}")

    # Tampilkan probabilitas untuk setiap kelas dalam format yang lebih rapi
    print("\nProbability for each class:")
    for i, class_name in enumerate(class_names):
        print(f"  {class_name}: {predictions[0][i]:.4f}")

    print("-" * 60)

In [ ]:
# Simpan model
model.save('/content/drive/MyDrive/model_final.h5')
print("Model saved successfully!")